# DataFrame Operations: Batch Processing

This notebook demonstrates how to work with multiple sequences efficiently using pandas DataFrames. The DataFrame-level functions in `seq_tools` are optimized for batch operations.

## Creating DataFrames from Sequences

First, let's see how to create DataFrames from sequences.


In [ ]:
from seq_tools import sequences_to_dataframe, sequence_to_dataframe
import pandas as pd

# Create DataFrame from multiple sequences
sequences = ["ATCGATCG", "GCTAGCTA", "AAAAATTTTT"]
names = ["seq1", "seq2", "seq3"]
df = sequences_to_dataframe(sequences, names=names)

print("DataFrame created from sequences:")
print(df)

## Basic Sequence Operations

### Adding Sequences to 5' and 3' Ends

Add constant sequences (like adapters or primers) to all sequences in a DataFrame.


In [ ]:
from seq_tools import add

# Add 5' and 3' sequences (e.g., adapters)
df_with_adapters = add(df, p5_seq="AAAA", p3_seq="TTTT")
print("Original sequences:")
print(df[["name", "sequence"]])
print("\nWith 5' and 3' adapters:")
print(df_with_adapters[["name", "sequence"]])

### Trimming Sequences

Remove nucleotides from the 5' and/or 3' ends of sequences.


In [ ]:
from seq_tools import trim

# Trim 2 bases from 5' end and 1 base from 3' end
df_trimmed = trim(df, p5_length=2, p3_length=1)
print("Original sequences:")
print(df[["name", "sequence"]])
print("\nTrimmed (2 from 5', 1 from 3'):")
print(df_trimmed[["name", "sequence"]])

## Sequence Conversions

### Converting Between DNA and RNA


In [ ]:
from seq_tools import to_rna_df, to_dna_df, to_dna_template_df

# Convert DNA to RNA
df_rna = to_rna_df(df)
print("DNA sequences:")
print(df[["name", "sequence"]])
print("\nConverted to RNA:")
print(df_rna[["name", "sequence"]])

# Convert back to DNA
df_dna = to_dna_df(df_rna)
print("\nConverted back to DNA:")
print(df_dna[["name", "sequence"]])

# Convert to DNA template (includes T7 promoter and reverse complement)
df_template = to_dna_template_df(df_rna)
print("\nDNA template (with T7 promoter):")
print(df_template[["name", "sequence"]])

### Transcription

Transcribe DNA templates to RNA. This is particularly useful for sequences with T7 promoters.


In [ ]:
from seq_tools import transcribe

# Create DNA template sequences with T7 promoter
# Note: T7 promoter is 20 bases, so we'll add it to our sequences
from seq_tools.config import T7_PROMOTER

templates = [T7_PROMOTER + seq for seq in sequences]
df_template = sequences_to_dataframe(templates, names=names)

# Transcribe to RNA (removes T7 promoter and folds)
df_transcribed = transcribe(df_template)
print("DNA templates:")
print(df_template[["name", "sequence"]])
print("\nTranscribed to RNA (T7 promoter removed, folded):")
print(df_transcribed[["name", "sequence", "structure", "mfe"]])

## Batch Folding

Fold multiple RNA sequences at once.


In [ ]:
from seq_tools import fold

# Convert sequences to RNA first
rna_seqs = ["GGGGUUUUCCCC", "GCGAAAGC", "AUGCAUGCAUGC"]
df_rna = sequences_to_dataframe(
    rna_seqs, names=[f"rna_{i+1}" for i in range(len(rna_seqs))]
)

# Fold all sequences
df_folded = fold(df_rna)
print("Folded RNA sequences:")
print(df_folded[["name", "sequence", "structure", "mfe", "ens_defect"]])

## Calculating Properties

### Molecular Weight


In [ ]:
from seq_tools import get_molecular_weight_df, determine_ntype

# Determine nucleotide type
ntype = determine_ntype(df)
print(f"Detected nucleotide type: {ntype}")

# Calculate molecular weights
df_mw = get_molecular_weight_df(df, ntype=ntype, double_stranded=False)
print("\nSequences with molecular weights:")
print(df_mw[["name", "sequence", "molecular_weight"]])

### Extinction Coefficient


In [ ]:
from seq_tools import get_extinction_coeff_df

# Calculate extinction coefficients
df_ec = get_extinction_coeff_df(df_folded, ntype="RNA", double_stranded=False)
print("Sequences with extinction coefficients:")
print(df_ec[["name", "sequence", "extinction_coeff"]])

### Reverse Complement


In [ ]:
from seq_tools import get_reverse_complement_df

# Calculate reverse complements
df_rc = get_reverse_complement_df(df, ntype="DNA")
print("Sequences with reverse complements:")
print(df[["name", "sequence"]])
print("\nReverse complements:")
print(df_rc[["name", "sequence"]])

### Sequence Length


In [ ]:
from seq_tools import get_length

# Get sequence lengths
df_len = get_length(df)
print("Sequences with lengths:")
print(df_len[["name", "sequence", "length"]])

## Sequence Pattern Matching

Check if sequences contain specific patterns or structures.


In [ ]:
from seq_tools import has_sequence, has_5p_sequence, has_3p_sequence, has_t7_promoter

# Check for sequences containing a pattern
search_pattern = "ATCG"
print(f"Checking for pattern '{search_pattern}':")
for idx, row in df.iterrows():
    result = has_sequence(df.iloc[[idx]], search_pattern)
    print(f"  {row['name']}: {result}")

# Check for 5' sequence
print(f"\nSequences starting with 'AT':")
for idx, row in df.iterrows():
    result = has_5p_sequence(df.iloc[[idx]], "AT")
    print(f"  {row['name']}: {result}")

# Check for 3' sequence
print(f"\nSequences ending with 'CG':")
for idx, row in df.iterrows():
    result = has_3p_sequence(df.iloc[[idx]], "CG")
    print(f"  {row['name']}: {result}")

## File I/O

### Export to FASTA Format


In [ ]:
from seq_tools import to_fasta
import os

# Export to FASTA
output_file = "example_output.fasta"
to_fasta(df, output_file)

# Display the file contents
if os.path.exists(output_file):
    with open(output_file, "r") as f:
        print("FASTA file contents:")
        print(f.read())
    # Clean up
    os.remove(output_file)

## Summary

In this notebook, we've covered:

- ✅ Creating DataFrames from sequences
- ✅ Adding sequences to 5' and 3' ends
- ✅ Trimming sequences
- ✅ Converting between DNA and RNA
- ✅ Transcribing DNA templates
- ✅ Batch folding RNA sequences
- ✅ Calculating molecular weights, extinction coefficients, and lengths
- ✅ Pattern matching in sequences
- ✅ Exporting to FASTA format

Next, we'll explore advanced features in **05_advanced_features.ipynb**.
